# Promotional Discount Elasticity Exploration with PROC PLOT

## Executive Summary

A retail merchandising analyst uses PROC PLOT to produce quick character-based scatter plots that explore how promotional discount depth and advertising spend drive weekly unit sales across three consumer-goods categories. The notebook generates a synthetic weekly SKU-level promotion dataset inline (36 rows: 3 categories × 12 weeks), then exercises PROC PLOT's core PLOT statement together with plotting symbols, reference lines, axis control, an overlay, and per-category subsetting to surface the price-elasticity and category-baseline patterns.

The executed listings below confirm a clear ordering of category demand — Beverages highest, Snacks in the middle, Household lowest — and a visible upward drift in units as discount depth increases. The advertising overlay shows the revenue series running above the units series across the spend range, a direct consequence of the dollar revenue scale. Every figure is the real listing produced by the run.

## Data Sources

**Synthetic dataset: `promo_weeks`** — one row per category per promotional week (36 rows = 3 categories × 12 weeks), generated inline with `call streaminit(20250529)` and `rand()`. No external or network inputs.

| Variable | Type | Description |
|----------|------|-------------|
| `category` | char | Product category (`Snacks`, `Beverages`, `Household`) |
| `week` | num | Promotion week index (1-12) |
| `discount_pct` | num | Promotional discount depth, percent off list (5-40) |
| `ad_spend` | num | Weekly advertising spend for the SKU, USD (200-3000) |
| `base_price` | num | List (undiscounted) unit price, USD |
| `units_sold` | num | Weekly units sold (responds to discount + ad spend) |
| `revenue` | num | Weekly net revenue, USD = units_sold * base_price * (1 - discount_pct/100) |

# Promotional Discount Elasticity Exploration with PROC PLOT

A merchandising analyst at a multi-category retail chain wants a fast, dependency-free way to eyeball how **promotional discount depth** and **advertising spend** move **weekly unit sales** — before committing to a formal regression. `PROC PLOT` is ideal for this: it draws character-based scatter plots straight to the listing, so the patterns (price elasticity, category baselines) jump out without any graphics device.

This notebook:
1. Generates a synthetic weekly promotion dataset inline.
2. Walks through `PROC PLOT`'s principal features — plotting symbols, reference lines, axis control, a two-series overlay, and per-category subsetting — exactly as a SAS analyst would for this question.

## Step 1 — Generate synthetic weekly promotion data

We simulate twelve promotional weeks for SKUs in three categories (Snacks, Beverages, Household). Units sold rise with **discount depth** (price elasticity) and, with diminishing returns, with **advertising spend**, plus category-specific baselines and random noise. Net `revenue` reflects the discounted price actually charged.

In [1]:
data promo_weeks;
    call streaminit(20250529);
    length category $9;
    array cats[3] $9 _temporary_ ('Snacks' 'Beverages' 'Household');
    array baseprice[3] _temporary_ (3.49 1.99 8.99);
    array baseunits[3] _temporary_ (140 220 60);
    do c = 1 to 3;
        category   = cats[c];
        base_price = baseprice[c];
        do week = 1 to 12;
            /* Discount depth 5%-40%, advertising spend $200-$3000 */
            discount_pct = round(5 + 35 * rand('uniform'), 1);
            ad_spend     = round(200 + 2800 * rand('uniform'), 10);

            /* Demand: elastic response to discount + diminishing ad lift + noise */
            elasticity_lift = baseunits[c] * (discount_pct / 100) * 2.2;
            ad_lift         = 18 * log(ad_spend / 200);
            noise           = rand('normal', 0, 8);
            units_sold      = round(baseunits[c] + elasticity_lift
                                     + ad_lift + noise, 1);
            if units_sold < 0 then units_sold = 0;

            revenue = round(units_sold * base_price
                            * (1 - discount_pct / 100), 0.01);
            output;
        end;
    end;
    drop c elasticity_lift ad_lift noise;
run;

NOTE: DATA promo_weeks


NOTE: Wrote promo_weeks (36 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.03 seconds
  cpu   0.03 seconds


## Step 2 — Units sold versus discount depth

The central merchandising question is the **price-elasticity relationship**: as the discount deepens, do units accelerate? We plot `units_sold` against `discount_pct` and draw a `BOX` around the plot for a clean frame.

In [2]:
proc plot data=promo_weeks;
    plot units_sold * discount_pct / box;
run;

                      Plot of UNITS_SOLD*DISCOUNT_PCT                      

     +----------------------------------------------------------------------+
500 |                                                                      |
    |                                                                      |
    |                                                                      |
    |                                                         1 1          |
400 |                                                     1                |
    |                                         1                            |
    |                                 1 1                                  |
    |                               1 1                                    |
300 |                2    2                                               1|
    |                                         1           1            11  |
    |                                 1                                    

## Step 3 — Distinguish categories with a plotting symbol

The blended cloud above mixes three categories with very different baselines. We use the `=category` form of the plot request so each point is drawn with the **first character of its category** as the plotting symbol, separating the three demand curves on a single set of axes.

In [3]:
proc plot data=promo_weeks;
    plot units_sold * discount_pct = category / box;
run;

                      Plot of UNITS_SOLD*DISCOUNT_PCT                      
     Symbol is value of CATEGORY.

     +----------------------------------------------------------------------+
500 |                                                                      |
    |                                                                      |
    |                                                                      |
    |                                                         B B          |
400 |                                                     B                |
    |                                         B                            |
    |                                 B B                                  |
    |                               B B                                    |
300 |                B    B                                               S|
    |                                         S           S            SS  |
    |                                 S  

## Step 4 — Reference lines and explicit axis scaling

Merchandising guidelines flag any promotion deeper than **25%** as margin-eroding, and the category target is roughly **200 units/week**. We add a vertical reference line at the 25% discount threshold (`HREF=`) and a horizontal reference line at the 200-unit target (`VREF=`), and we pin both axes to fixed tick marks (`HAXIS=`, `VAXIS=`) so successive runs stay comparable.

In [4]:
proc plot data=promo_weeks;
    plot units_sold * discount_pct = category /
         haxis = 0 to 40 by 5
         vaxis = 0 to 400 by 50
         href  = 25
         vref  = 200
         box;
run;

                      Plot of UNITS_SOLD*DISCOUNT_PCT                      
     Symbol is value of CATEGORY.

     +----------------------------------------------------------------------+
400 |                                                                      |
    |                                         B                            |
    |                                   B                                  |
350 |                                 B                                    |
    |                     B         B                                      |
300 |                B                                                    S|
    |                                                     S            SS  |
    |                                         S                            |
250 |                                 S         S      S                   |
    |                 S                                                    |
    |         S            S             

## Step 5 — Advertising spend: revenue and units on one frame

Does advertising spend track with returns? We `OVERLAY` two plots against `ad_spend` on a shared axis — `revenue` (symbol `R`) and `units_sold` (symbol `U`) — to compare their levels and spread on a single frame. Because revenue is measured in dollars and units in counts, the two clouds occupy different vertical bands, which is exactly what the overlay is meant to reveal at a glance.

In [5]:
proc plot data=promo_weeks;
    plot revenue    * ad_spend = 'R'
         units_sold * ad_spend = 'U' / overlay box;
run;

               Plot of REVENUE*AD_SPEND, UNITS_SOLD*AD_SPEND                
      Symbol used is '*'.

      +----------------------------------------------------------------------+
1200 |                                                                      |
     |                                                                      |
     |                                                                      |
     |                                                          R           |
1000 |                                            R                R        |
     |                                           R                          |
     |                       R                                              |
 800 |                       R                    R                         |
     |               RR                                    R       R        |
     |     R        R     R                R             R           R      |
     |       R                R      

## Step 6 — Isolate a single category's elasticity

The combined plots mix three very different demand baselines. To read one category's elasticity slope on its own, we subset with a `WHERE` clause and replot `units_sold * discount_pct` for **Beverages** alone — the highest-volume category. Restricting to a single category removes the baseline offset so the within-category relationship between discount depth and units stands out clearly.

In [6]:
proc plot data=promo_weeks;
    where category = 'Beverages';
    plot units_sold * discount_pct / box;
run;

                      Plot of UNITS_SOLD*DISCOUNT_PCT                      

     +----------------------------------------------------------------------+
450 |                                                                      |
    |                                                                      |
    |                                                                   1  |
    |                                                                      |
    |                                                                1     |
    |                                                            1         |
    |                                                                      |
400 |                                                                      |
    |                                                                      |
    |                                                                      |
    |                                            1                         

## Interpreting the results

- **Discount elasticity (Steps 2-4).** Deeper discounts pull unit volume up in a broad upward band — consistent with meaningful price sensitivity. The `=category` symbol plot cleanly separates the three baselines: across the 12 simulated weeks, **Beverages** sit highest (mean ≈ 357 units/week), **Snacks** in the middle (≈ 250), and **Household** lowest (≈ 122). The 25% `HREF` and 200-unit `VREF` lines in Step 4 partition the frame so you can read at a glance which points cleared the 200-unit target and which fell on the deep-discount (>25%) side of the margin-erosion threshold — every Beverages and Snacks point plots above the 200 line, while every Household point sits below it.
- **Advertising spend (Step 5).** The overlay places `revenue` (`R`) and `units_sold` (`U`) on one `ad_spend` axis. The `R` cloud spans roughly the 580-1050 band and the `U` cloud roughly 100-450, so revenue plots above units across the whole spend range — a scale effect (dollars vs. counts), not a causal claim. Both clouds are wide and show no strong tilt against `ad_spend`, so at this sample size the listing reveals no clear spend-to-outcome gradient; that flatness is itself a useful "look before you model" signal.
- **Single-category view (Step 6).** Subsetting to Beverages with `WHERE` strips out the cross-category baseline offset, leaving a tighter `units_sold * discount_pct` cloud (≈ 305-440 units) whose upward tilt is the within-category elasticity slope — the relationship a follow-up model would quantify.

**Caveat:** `PROC PLOT` is an *exploratory* tool — it reveals shape and direction, not significance, and with only 12 weeks per category these clouds are sparse. The patterns here justify a follow-up `PROC REG` or `PROC GLM` to quantify the discount effect with confidence intervals before setting promotional policy.